<a href="https://colab.research.google.com/github/Shineii86/MoeStickerBot/blob/main/notebooks/MoeStickerBotV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Moe Sticker Bot

**Import LINE/Kakao stickers to Telegram · Create custom packs · Self-host in Colab**

---

### Quick Start

1. Run the setup cell below.
2. Enter your bot token and run the launch cell.
3. Monitor logs or stop the bot with the last cell.

In [ ]:
#@title 1. Setup Environment & Build Bot

import os, subprocess, urllib.request, sys, time

print("Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2

print("Downloading Go...")
url = "https://go.dev/dl/go1.21.5.linux-amd64.tar.gz"
urllib.request.urlretrieve(url, "go.tar.gz")
!tar -C /usr/local -xzf go.tar.gz
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
!mkdir -p $GOPATH

print("Installing Python helpers...")
!wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/msb_emoji.py -O /usr/local/bin/msb_emoji.py
!wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/msb_kakao_decrypt.py -O /usr/local/bin/msb_kakao_decrypt.py
!wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/msb_rlottie.py -O /usr/local/bin/msb_rlottie.py
!chmod +x /usr/local/bin/*.py

print("Building bot...")
!rm -rf MoeStickersBot
!git clone --depth 1 https://github.com/Shineii86/MoeStickersBot.git
%cd MoeStickersBot
!go mod download
!go build -o MoeStickersBot cmd/MoeStickersBot/main.go

if os.path.exists("MoeStickersBot"):
    print("Build successful!")
else:
    print("Build failed.")


In [ ]:
#@title 2. Configure & Launch Bot

BOT_TOKEN = ""  #@param {type:"string"}
ENABLE_DB = False  #@param {type:"boolean"}
DB_ADDR = "localhost:3306"  #@param {type:"string"}
DB_USER = "moe_bot"  #@param {type:"string"}
DB_PASS = ""  #@param {type:"string"}
DB_NAME = "moe_sticker_bot"  #@param {type:"string"}
ENABLE_WEBAPP = False  #@param {type:"boolean"}
WEBAPP_PORT = 8080  #@param {type:"integer"}
NGROK_AUTHTOKEN = ""  #@param {type:"string"}
DATA_DIR = "moe_sticker_bot_data"  #@param {type:"string"}
LOG_LEVEL = "info"  #@param ["debug", "info", "warn", "error"]
HTTP_PROXY = ""  #@param {type:"string"}

if not BOT_TOKEN:
    print("ERROR: Please provide a Bot Token.")
    sys.exit(1)

cmd = ["./MoeStickersBot", f"--bot_token={BOT_TOKEN}", f"--log_level={LOG_LEVEL}", f"--data_dir={DATA_DIR}"]
if ENABLE_DB and DB_ADDR:
    cmd.extend([f"--db_addr={DB_ADDR}", f"--db_user={DB_USER}", f"--db_pass={DB_PASS}", f"--db_name={DB_NAME}"])
if HTTP_PROXY:
    cmd.append(f"--http_proxy={HTTP_PROXY}")

if ENABLE_WEBAPP and NGROK_AUTHTOKEN:
    print("WebApp requires manual ngrok setup. Skipping.")

print("Starting bot...")
log_out = open("bot_stdout.log", "w")
log_err = open("bot_stderr.log", "w")
process = subprocess.Popen(cmd, stdout=log_out, stderr=log_err)
time.sleep(5)
if process.poll() is None:
    print(f"Bot is running (PID {process.pid}). Send /start on Telegram!")
else:
    print("Bot exited. Check bot_stderr.log")


In [ ]:
#@title 3. Monitor & Control

ACTION = "View Logs"  #@param ["View Logs", "Stop Bot"]
LOG_TYPE = "stderr"  #@param ["stdout", "stderr"]
LINES = 30  #@param {type:"slider", min:10, max:100, step:10}

if ACTION == "View Logs":
    !tail -n {LINES} bot_{LOG_TYPE}.log
else:
    !pkill -f MoeStickersBot
    print("Bot stopped.")
